In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widget de saída para exibir mensagens e gráficos
output_display = widgets.Output()

def analisar(b):
    with output_display:
        clear_output() # Limpa saídas anteriores do widget
        ativo = text_input.value.strip()

        if ativo == "":
            print('Erro: Digite um ativo!')
            return

        try:
            # Download dos dados
            dados = yf.download(ativo, start='2020-01-01')

            if dados.empty:
                print('Erro: Ativo não encontrado ou sem dados!')
                return

            # Cálculos de Médias Móveis
            dados['MM20'] = dados['Close'].rolling(20).mean()
            dados['MM50'] = dados['Close'].rolling(50).mean()

            # Estratégia de Sinais
            dados['Sinal'] = 0
            # np.where para definir a condição de cruzamento
            dados.iloc[50:, dados.columns.get_loc('Sinal')] = np.where(
                dados['MM20'][50:] > dados['MM50'][50:], 1, 0
            )

            # Retornos
            dados['Retorno'] = dados['Close'].pct_change()
            dados['Retorno_Estrategia'] = (dados['Retorno'] * dados['Sinal'].shift(1))
            retorno_acumulado = (1 + dados['Retorno_Estrategia']).cumprod()

            # Gráfico 1: Preços e Médias
            plt.figure(figsize=(10, 5))
            plt.plot(dados['Close'], label='Preço')
            plt.plot(dados['MM20'], label='MM20')
            plt.plot(dados['MM50'], label='MM50')
            plt.legend()
            plt.title(f'Análise de {ativo}')
            plt.show()

            # Gráfico 2: Retorno da Estratégia
            plt.figure(figsize=(10, 5))
            plt.plot(retorno_acumulado)
            plt.title('Retorno Acumulado da Estratégia')
            plt.show()

        except Exception as e:
            print(f'Erro: Falha ao processar os dados. Verifique se o código do ativo está correto. Detalhes: {e}')

# --- Configuração dos Widgets do Jupyter ---

text_input = widgets.Text(
    value='',
    placeholder='Ex: PETR4.SA',
    description='Código do ativo:',
    disabled=False
)

analyze_button = widgets.Button(
    description='Analisar',
    disabled=False,
    button_style='info', # Cor do botão
    tooltip='Clique para analisar o ativo',
    icon='chart-line' # Ícone (FontAwesome)
)

# Conecta a função 'analisar' ao evento de clique do botão
analyze_button.on_click(analisar)

# Exibe os widgets e o output
display(text_input, analyze_button, output_display)

Text(value='', description='Código do ativo:', placeholder='Ex: PETR4.SA')

Button(button_style='info', description='Analisar', icon='chart-line', style=ButtonStyle(), tooltip='Clique pa…

Output()